<a href="https://colab.research.google.com/github/farrelrassya/python-for-finance/blob/main/ch06_Object_Oriented_Programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6 — Object-Oriented Programming

> *"The purpose of software engineering is to control complexity, not to create it."* — Pamela Zave

Object-oriented programming (**OOP**) is the dominant paradigm in Python — so dominant that the saying *"everything in Python is an object"* is literally true: integers, functions, modules, even classes themselves are objects with attributes and methods. For a quant or ML engineer, fluency with OOP is the difference between **using** libraries (calling `model.fit(X, y)`) and **extending** them (writing a custom estimator, a PyTorch `nn.Module`, or a derivative-pricing class hierarchy).

This chapter develops the concepts and Python syntax around six pillars:

1. **Classes & instances** — abstract templates and concrete realizations.
2. **Attributes & methods** — the data and behavior that belong to objects.
3. **Inheritance** — sharing behavior across class hierarchies.
4. **Aggregation & composition** — objects built from other objects.
5. **Polymorphism (duck typing)** — letting different types share an interface.
6. **The Python data model** — the protocol of `__init__`, `__repr__`, `__add__`, `__len__`, `__iter__`, etc., that lets your custom classes integrate seamlessly with the rest of Python.

### Why this matters for ML/quant work

The libraries we use daily are deeply object-oriented: scikit-learn's `BaseEstimator` interface, PyTorch's `nn.Module`, pandas' `DataFrame` — every one of them earns its ergonomics through the data model. The same techniques let *you* build a `Strategy`, `Portfolio`, or `Model` class whose instances behave like first-class Python citizens — hashable, iterable, composable, printable.

### Vocabulary preview

| Term | Meaning |
|---|---|
| **Class** | An abstract definition of a *type* of object (e.g., `HumanBeing`, `Stock`). |
| **Object** / **Instance** | A concrete realization of a class (e.g., `sandra = HumanBeing(...)`). |
| **Attribute** | Data attached to a class or instance (e.g., `sandra.eye_color`). |
| **Method** | A function attached to a class — receives `self` (the instance) as first argument. |
| **Instantiation** | The act of creating an instance: `sandra = HumanBeing('Sandra', 'blue')`. |

We'll build up these ideas in order: first looking at OOP that already surrounds us in Python, then writing classes from scratch, and finally implementing a `Vector` class that integrates with Python's full operator set.


## Setup

For this chapter we use only `numpy` and `pandas` — the OOP work itself is pure Python.


In [1]:
# Standard imports for the chapter
import numpy as np
import pandas as pd
import sys
import warnings
warnings.filterwarnings('ignore')

print(f"Python:    {sys.version.split()[0]}")
print(f"numpy:     {np.__version__}")
print(f"pandas:    {pd.__version__}")

Python:    3.12.13
numpy:     2.0.2
pandas:    2.2.2


These versions are recorded for reproducibility. The Python language version is especially relevant for OOP — features like `match`/`case` (3.10+), `dataclasses` (3.7+), and PEP-695 generic syntax (3.12+) all live in the language proper, not in any library.


## 1. A First Example — `HumanBeing`

Before defining new vocabulary, let's see all six pillars in a tiny class.


In [2]:
class HumanBeing(object):
    """A trivial class with two attributes and one method."""

    def __init__(self, first_name, eye_color):
        # __init__ is called automatically during instantiation.
        # `self` is the instance being built; the other parameters
        # come from the call site.
        self.first_name = first_name      # instance attribute
        self.eye_color = eye_color        # instance attribute
        self.position = 0                 # default starting position

    def walk_steps(self, steps):
        """Move the human forward by `steps` positions."""
        self.position += steps

# Instantiate: 'Sandra' and 'blue' are passed to __init__
Sandra = HumanBeing('Sandra', 'blue')

# Inspect attributes
print(f"first_name: {Sandra.first_name}")
print(f"eye_color:  {Sandra.eye_color}")
print(f"position:   {Sandra.position}")

# Call a method — this mutates state
Sandra.walk_steps(5)
print(f"after walk_steps(5), position: {Sandra.position}")

first_name: Sandra
eye_color:  blue
position:   0
after walk_steps(5), position: 5


Three observations about this minimal example:

**1. The `self` convention.** Every method's first parameter is conventionally named `self` — it is the *instance* the method is operating on. When we call `Sandra.walk_steps(5)`, Python translates this to `HumanBeing.walk_steps(Sandra, 5)`, automatically supplying `Sandra` as `self`. The transformation is purely syntactic; `self` is not a keyword, just a convention. Some old codebases use `this` or `_`; Python's PEP 8 and 99% of code use `self`, so always do too.

**2. `__init__` is the constructor.** The double underscores ("dunder") mark this as a **special method** — Python invokes it automatically during instantiation. Inside `__init__` we set instance attributes via `self.X = value`. These attributes belong to *this specific instance* (`Sandra`), not to all `HumanBeing`s collectively. Each call to `HumanBeing(...)` produces a fresh object with its own attribute storage.

**3. State mutation through methods.** `walk_steps(5)` doesn't return anything visible — it modifies `self.position` in place, taking it from $0$ to $5$. This is the *encapsulated state* pattern: an object holds data, and exposed methods are the legitimate way to change it. We'll see in section 4 how to make this discipline more enforceable via "private" attributes.

**Connection to ML libraries:** every scikit-learn estimator follows exactly this template: `__init__` stores hyperparameters, `.fit(X, y)` mutates the instance to record learned state (`self.coef_`, `self.classes_`), `.predict(X)` reads that state and returns a value. Once you understand the pattern in `HumanBeing`, you understand `LogisticRegression`.


## 2. Why OOP? Six Technical Drivers

Six concepts dominate the OOP lexicon. Each one solves a real problem you'll hit when modeling complex systems.

**Abstraction** — describe *what* an object is and does, not *how* it's stored. A `FinancialInstrument` class hides whether the underlying data lives in a database, a flat file, or a remote API.

**Modularity** — break a problem into independent classes. A European call option is naturally split into the *underlying* (a stock with a price process) and the *option* (with strike, maturity, and a payoff function).

**Inheritance** — reuse behavior by extending classes. `EuropeanCallOption` $\subset$ `EuropeanOption` $\subset$ `Derivative` $\subset$ `FinancialInstrument`. Adding a new option type might mean writing only the `payoff` method while inheriting everything else.

**Aggregation vs. composition** — *aggregation* relates objects with independent lifetimes (a swap holds a reference to a yield curve that exists separately). *Composition* relates objects with linked lifetimes (an interest-rate swap's two legs cannot exist outside the swap).

**Polymorphism (duck typing)** — different classes can implement the same method (e.g., `get_current_price()`) and be used interchangeably. Python's *"if it walks like a duck and quacks like a duck, it's a duck"* philosophy means polymorphism doesn't require explicit interfaces — same method names, same return types, that's enough.

**Encapsulation** — hide state behind methods so external code doesn't accidentally corrupt it. In Python this is more of a *convention* than a hard rule (we'll see why in section 4).

These map directly to ML and quant patterns:

| OOP concept | scikit-learn | PyTorch |
|---|---|---|
| Inheritance | `BaseEstimator`, `ClassifierMixin` | `nn.Module` |
| Composition | `Pipeline`, `ColumnTransformer` | `nn.Sequential`, sub-modules |
| Polymorphism | every estimator implements `.fit(X, y)` | every module implements `.forward(x)` |
| Encapsulation | `_private` learned attributes | `_modules`, `_parameters` dicts |


## 3. A Look at Python Objects

Before writing our own classes, let's look at the OOP machinery already around us. Every value in Python — even an integer literal `5` — is an object with a class, attributes, and methods.

### 3.1 `int`

The integer `5` is an instance of class `int`. It has attributes (like `numerator` and `denominator`, reflecting that `int` inherits from the abstract `Rational` type), methods (like `bit_length()`), and works with operators (`+`, `*`) — which are themselves dispatched to special methods (`__add__`, `__mul__`).


In [3]:
# A garden-variety integer
n = 5

# Every Python value has a type — this is its class
print(f"type(n):           {type(n)}")

# int exposes useful attributes from the numbers.Integral hierarchy
print(f"n.numerator:       {n.numerator}")

# Methods reveal how int is implemented internally
print(f"n.bit_length():    {n.bit_length()}    # bits needed to represent 5 in binary: 101")

# Operators dispatch to special methods on the class
print(f"n + n:             {n + n}              # calls n.__add__(n)")
print(f"2 * n:             {2 * n}              # calls n.__mul__(2)")

# Even __sizeof__ — the memory footprint — is a method
print(f"n.__sizeof__():    {n.__sizeof__()} bytes")

type(n):           <class 'int'>
n.numerator:       5
n.bit_length():    3    # bits needed to represent 5 in binary: 101
n + n:             10              # calls n.__add__(n)
2 * n:             10              # calls n.__mul__(2)
n.__sizeof__():    28 bytes


Three things to take away from this innocuous-looking output:

**1. Integers occupy 28 bytes in CPython.** Far more than the 8 bytes you might expect for a 64-bit integer. The overhead pays for: a type pointer, a reference count (Python uses reference counting for memory management), and the variable-length digit array that lets Python integers be arbitrarily large (no overflow at $2^{63}$, unlike C). The cost: $5 \to 28$ bytes is a $3.5\times$ overhead per scalar. **This is why NumPy exists** — `np.int64` strips that overhead and packs values into contiguous arrays.

**2. `bit_length()` returns 3.** Because $5_{10} = 101_2$ — three bits suffice. The general formula:

$$
\texttt{bit\_length}(n) = \lfloor \log_2 |n| \rfloor + 1, \quad n \ne 0
$$

For $n = 5$: $\lfloor \log_2 5 \rfloor + 1 = 2 + 1 = 3$. ✓

**3. Operators are syntactic sugar for method calls.** Writing `n + n` is identical to calling `n.__add__(n)`. Python looks up `__add__` on the *type* (not the instance), invokes it, and returns the result. This indirection is what makes Python extensible: by defining `__add__` on your own class, your class gains the `+` operator. We'll do exactly this with the `Vector` class later.


### 3.2 `list`

A `list` is also an object — with more methods than `int` because it's a container.


In [4]:
l = [1, 2, 3, 4]

print(f"type(l):           {type(l)}")
print(f"l[0]:              {l[0]}              # __getitem__ in disguise")

# Methods that mutate the list in place
l.append(10)
print(f"after l.append(10): {l}")

# Operator overloading: + concatenates, * repeats
print(f"l + l:             {l + l}")
print(f"2 * l:             {2 * l}")

# Aggregating with a built-in function
print(f"sum(l):            {sum(l)}")

# Memory footprint
print(f"l.__sizeof__():    {l.__sizeof__()} bytes")

type(l):           <class 'list'>
l[0]:              1              # __getitem__ in disguise
after l.append(10): [1, 2, 3, 4, 10]
l + l:             [1, 2, 3, 4, 10, 1, 2, 3, 4, 10]
2 * l:             [1, 2, 3, 4, 10, 1, 2, 3, 4, 10]
sum(l):            20
l.__sizeof__():    104 bytes


Notice how `list` overloads operators in **container-natural ways**: `+` is *concatenation* (not element-wise addition), `*` is *repetition* (not element-wise multiplication). The same operator means different things on different types — this is **operator polymorphism**, the cleanest example of why Python's data model is powerful.

The memory footprint deserves attention. The list `[1, 2, 3, 4, 10]` reports `__sizeof__() = 104` bytes. Where does this go?

$$
\text{list overhead} \;+\; \underbrace{N \times \text{pointer size}}_{\text{slots for object refs}} \;+\; \text{growth slack}
$$

CPython's list is an *array of pointers*, and to amortize the cost of `append`, it over-allocates capacity. Adding one element to a length-4 list typically grows capacity to 8, leaving slack. Crucially, `__sizeof__()` measures only the list itself — **not** the integer objects it points to. The five `int`s pointed at by `l` add another $5 \times 28 = 140$ bytes, making the *real* footprint about $244$ bytes for what stores 5 small integers.

**ML/quant takeaway:** for any nontrivial numeric work, prefer NumPy or pandas. A list of one million Python `int`s consumes roughly $36$ MB; a NumPy `int64` array of the same length is $8$ MB — a $4.5\times$ memory advantage *and* vastly faster operations because of vectorization. Native Python collections are for small, heterogeneous, structural data; arrays are for numbers.


### 3.3 `numpy.ndarray` — a custom-built object that *acts* native

`numpy.ndarray` is not a built-in type — it's defined in the NumPy package. Yet it integrates with Python operators (`+`, `*`), built-in functions (`sum`, `len`), and special methods (`__sizeof__`) as if it were native. **This is the Python data model paying dividends.**


In [5]:
# Build a 4x4 matrix of integers 0..15
a = np.arange(16).reshape((4, 4))
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15]])

A $4 \times 4$ matrix containing the integers $0, 1, \ldots, 15$ laid out row-by-row. The class:


In [6]:
print(f"type(a):     {type(a)}")
print(f"a.dtype:     {a.dtype}")
print(f"a.shape:     {a.shape}")
print(f"a.nbytes:    {a.nbytes}    # 16 elements × 8 bytes (int64) = 128")

type(a):     <class 'numpy.ndarray'>
a.dtype:     int64
a.shape:     (4, 4)
a.nbytes:    128    # 16 elements × 8 bytes (int64) = 128


The class is `numpy.ndarray` — defined in the `numpy` package, not built into Python itself. Yet despite being "third-party" code, it integrates with every Python operator and built-in function. The arithmetic is precise:

$$
\text{nbytes} = N_{\text{elements}} \times \text{itemsize} = 16 \times 8 = 128
$$

— exactly as reported. There's no per-element overhead at all, in contrast to `int` (28 bytes each) and `list` (8-byte pointer each plus the int it points to). This compactness, plus the fact that the data lives in a *single contiguous block of C memory*, is what makes NumPy fast: SIMD instructions, cache-friendly access patterns, and BLAS routines all become available.


In [7]:
# Aggregating method (collapses to a scalar)
print(f"a.sum():     {a.sum()}")

# Cumulative method (preserves shape, returns a new array)
print("a.cumsum(axis=0):")
print(a.cumsum(axis=0))

a.sum():     120
a.cumsum(axis=0):
[[ 0  1  2  3]
 [ 4  6  8 10]
 [12 15 18 21]
 [24 28 32 36]]


Two distinct method types: an **aggregation** (`sum` $\to$ scalar) and a **transformation** (`cumsum(axis=0)` $\to$ new array). The sum $0 + 1 + \cdots + 15 = \frac{15 \cdot 16}{2} = 120$. ✓

`cumsum(axis=0)` runs cumulative sums *down columns* — row $i$ in the result holds $\sum_{k=0}^{i} a_{k, j}$ for each column $j$. The bottom row is exactly the column sums:

$$
[0+4+8+12,\; 1+5+9+13,\; 2+6+10+14,\; 3+7+11+15] = [24, 28, 32, 36]
$$

— which we'll see appear again in a moment as `sum(a)` (the row-wise sum, by built-in iteration over rows).


In [8]:
# + and * on arrays — element-wise, NOT concatenation
print("a + a:")
print(a + a)
print()
print("2 * a:")
print(2 * a)

a + a:
[[ 0  2  4  6]
 [ 8 10 12 14]
 [16 18 20 22]
 [24 26 28 30]]

2 * a:
[[ 0  2  4  6]
 [ 8 10 12 14]
 [16 18 20 22]
 [24 26 28 30]]


Now `+` means **element-wise addition** and `*` means **element-wise multiplication** — the *opposite* convention from `list`! Whether `a + a` doubles the values or concatenates the arrays depends on the type. NumPy's choice (element-wise) is the natural one for numerical work; if you want concatenation you use `np.concatenate`.

This is **operator polymorphism in its pure form**: the *same code* (`x + x`) does dramatically different things depending on what `x` is. Library authors lean into this — pandas operators do element-wise *with index alignment*, PyTorch operators do element-wise *with autograd tracking*. Always check what the operator means in the context of the type you're working with.


In [9]:
# Built-in sum() iterates over rows; np.sum() reduces over the whole array
print(f"sum(a):     {sum(a)}      # iterates over rows (axis 0), sums them as vectors")
print(f"np.sum(a):  {np.sum(a)}                # reduces the whole array to a scalar")
print()
print(f"a.__sizeof__():  {a.__sizeof__()} bytes  # array header + data buffer")

sum(a):     [24 28 32 36]      # iterates over rows (axis 0), sums them as vectors
np.sum(a):  120                # reduces the whole array to a scalar

a.__sizeof__():  128 bytes  # array header + data buffer


**A subtle but important distinction**: `sum(a)` and `np.sum(a)` give different answers. Why?

- `sum(a)` is the built-in: it iterates over `a` (treating it as a sequence of rows), starting from `0` and adding each row. So it returns the column sums $[24, 28, 32, 36]$ — a 1-D array, not a scalar.
- `np.sum(a)` is NumPy's reduction: it flattens by default and adds all 16 elements, returning the scalar $120$.

For 1-D inputs both give the same answer. For 2-D arrays they diverge. **Production lesson:** when summing arrays, *always* use `np.sum` or `arr.sum()` and pass the `axis` argument explicitly. Python's built-in `sum` on multi-dimensional arrays is technically correct but conceptually surprising, and a source of subtle bugs.

The reported $\texttt{\_\_sizeof\_\_} = 128$ bytes here matches the data buffer almost exactly — NumPy's array header is allocated separately and is what gives the small overhead seen in some Python versions.


### 3.4 `pandas.DataFrame` — a richer container

The same operator-overloading patterns extend to pandas. A `DataFrame` adds labels (columns and index) on top of a 2-D array, but the OOP machinery is identical.


In [10]:
# Wrap the array in a labeled DataFrame
df = pd.DataFrame(a, columns=list('abcd'))
print(f"type(df):  {type(df)}")
df

type(df):  <class 'pandas.core.frame.DataFrame'>


,a,b,c,d
0,0,1,2,3
1,4,5,6,7
2,8,9,10,11
3,12,13,14,15


The class is `pandas.core.frame.DataFrame` (or just `pandas.DataFrame` in modern pandas's flatter namespace). Notice that the DataFrame *contains* the NumPy array — but it's not the array. It's a richer object built around the array, adding `columns` and `index` metadata.

This is **composition**: a `DataFrame` is built from a NumPy array (the `.values` attribute) plus index/column metadata. Each component object exists independently — you can extract `df.values` and pass it to scikit-learn unchanged. The layered design is how pandas inherits NumPy's speed while adding ergonomics.


In [11]:
# Attributes
df.columns

Index(['a', 'b', 'c', 'd'], dtype='object')

`df.columns` is itself a custom object — a `pandas.Index` — not a Python list. The `dtype='object'` indicates these labels are stored as Python strings. The whole point of having a dedicated `Index` class is that it supports its own operations (set operations, sorting, intersection with other indices) that a plain list would not.

This is OOP at work: the abstract concept "labels for the columns of a DataFrame" deserves its own type because labels have specific behaviors (uniqueness checks, hashable lookup, alignment) that distinguish them from generic sequences.


In [12]:
# Aggregations
df.sum()

,0
a,24
b,28
c,32
d,36


Column-wise sums, returned as a `Series` indexed by column name. The values match the column-wise sums we computed by hand: $[24, 28, 32, 36]$. Unlike `np.sum(a)` (which flattens), `df.sum()` *preserves the labels* — pandas defaults to "aggregate down rows, give one value per column" because labeled output is more useful in an analytical workflow.


In [13]:
df.cumsum()

,a,b,c,d
0,0,1,2,3
1,4,6,8,10
2,12,15,18,21
3,24,28,32,36


Cumulative sum down each column, returned as a DataFrame with the same shape and labels. Identical numerically to `a.cumsum(axis=0)` but with the ergonomic gain that the result *carries its row index and column labels* — you don't lose context as you flow through a pipeline.


In [14]:
# Element-wise + on DataFrames — same convention as ndarray
df + df

,a,b,c,d
0,0,2,4,6
1,8,10,12,14
2,16,18,20,22
3,24,26,28,30


Element-wise doubling, identical to NumPy's behavior. **Crucially**, pandas adds an extra ingredient: *index and column alignment*. If we had added two DataFrames with different column orders or indices, pandas would have aligned by label first, then added — inserting `NaN` where labels don't match. This is invisible when the operands are identical but is the killer feature when combining heterogeneous data.


In [15]:
# Scalar multiplication: broadcasts
2 * df

,a,b,c,d
0,0,2,4,6
1,8,10,12,14
2,16,18,20,22
3,24,26,28,30


Scalar multiplication broadcasts: every cell is doubled. The implementation goes through `df.__mul__(2)`, which dispatches to NumPy on the underlying buffer, then re-wraps with the DataFrame's index and columns. The interplay of `__mul__` (for `df * 2`) and `__rmul__` (for `2 * df`) is invisible to the user but baked in to make this just work.


In [16]:
# np.sum on a DataFrame: dispatches via the data model, returns column sums (Series)
np.sum(df)

,0
a,24
b,28
c,32
d,36


Even when we call NumPy's `np.sum` on a DataFrame, the call is intercepted (via the `__array_function__` protocol) and dispatched to `df.sum()`. The result is a labeled Series, not a scalar. **This kind of cross-library polymorphism is what the Python data model enables**: a function from one library can operate on objects from another by following standard protocols.

The same protocol-based interop is why `torch.from_numpy(arr)` doesn't actually copy data, why `xarray.DataArray` works inside `np.mean`, and why scikit-learn estimators happily accept pandas DataFrames as `X`. Once you understand the protocols, the apparent magic of "this just works" reveals itself as well-defined contract negotiation.


In [17]:
# DataFrame memory footprint
print(f"df.__sizeof__():  {df.__sizeof__()} bytes")

df.__sizeof__():  260 bytes


The DataFrame's footprint is larger than the underlying ndarray's $128$ bytes because pandas adds the column `Index`, the row `Index`, and per-column dtype metadata. That overhead is **constant in $N$ rows** — for a 1-million-row DataFrame the per-column overhead is still bytes, while the data is megabytes. So pandas's "wrapping" cost is irrelevant at scale; what you pay for in memory is essentially identical to NumPy.


## 4. Basics of Python Classes — Building `FinancialInstrument`

Now we write classes from scratch. The example object: a financial instrument with a symbol and a price. We'll evolve it across four versions to illustrate one OOP concept per step.

### 4.1 The minimal class


In [18]:
# The smallest possible class — `pass` does nothing
class FinancialInstrument(object):
    pass

# Even this empty class can be instantiated
fi = FinancialInstrument()
print(f"type(fi):  {type(fi)}")
print(f"repr(fi):  {fi!r}")
print(f"str(fi):   {fi.__str__()}")

type(fi):  <class '__main__.FinancialInstrument'>
repr(fi):  <__main__.FinancialInstrument object at 0x78802da95160>
str(fi):   <__main__.FinancialInstrument object at 0x78802da95160>


**A class with `pass` is still a fully-functional Python class.** Every Python class — even an empty one — *inherits* from the base `object` class and acquires:

- `__init__`, `__str__`, `__repr__`, `__hash__`, `__eq__`, `__sizeof__`, `__class__`, ...

That's why `fi.__str__()` works without us writing it: `object.__str__` is inherited and produces the default `<__main__.ClassName object at 0xMEMADDR>` representation.

The hexadecimal address is the **id of the object in CPython** — essentially its memory location, useful only for distinguishing one instance from another. Two distinct `FinancialInstrument()` calls give different addresses; the same instance accessed twice gives the same address. We'll override `__repr__` later to give a more useful display.


In [19]:
# Python lets you attach attributes to instances on the fly — without prior declaration
fi.price = 100
print(f"fi.price:  {fi.price}")

fi.price:  100


**Dynamic attribute assignment** is a feature of Python classes that surprises programmers from Java/C++/C# backgrounds. There's no field declaration; you just assign and the attribute exists.

This flexibility is powerful but dangerous in production:

- **Powerful**: monkey-patching for tests, attaching metadata to objects, prototyping without ceremony.
- **Dangerous**: a typo (`fi.pirce = 100`) silently creates a *new* attribute instead of failing. The original `price` retains its old value. Hours of debugging later, you discover Python "helped" you create the wrong field.

For production, use **dataclasses** (`@dataclass` decorator), `__slots__`, or `pydantic` to declare expected fields and reject typos. We'll see one of these tools shortly.


### 4.2 Class attributes vs. instance attributes — and `__init__`

Class attributes are shared across all instances. Instance attributes are per-object. The distinction matters for memory and for "global" state.


In [20]:
class FinancialInstrument(object):
    # Class attribute — shared by every instance
    author = 'Yves Hilpisch'

    def __init__(self, symbol, price):
        # Instance attributes — unique to each instance
        self.symbol = symbol
        self.price = price

# Class attributes are accessible via the class itself (no instance needed)
print(f"Class attr (via class):     {FinancialInstrument.author}")

# Instantiate and inspect
aapl = FinancialInstrument('AAPL', 100)
print(f"Instance attr 'symbol':     {aapl.symbol}")
print(f"Instance attr 'price':      {aapl.price}")
print(f"Class attr (via instance):  {aapl.author}")

# Instance attributes can be reassigned freely
aapl.price = 105
print(f"After update, price:        {aapl.price}")

Class attr (via class):     Yves Hilpisch
Instance attr 'symbol':     AAPL
Instance attr 'price':      100
Class attr (via instance):  Yves Hilpisch
After update, price:        105


**The lookup rule for `aapl.author`:** Python first checks the instance's own `__dict__`. If `author` is not there (and it isn't — `__init__` didn't set it), Python falls back to the *class*'s `__dict__`. There it finds `author = 'Yves Hilpisch'` and returns that. This is Python's **MRO (Method Resolution Order)** at work — the same logic walks up the inheritance chain when subclasses are involved.

**Memory implication.** A class attribute is stored *once*, in the class. A million instances of `FinancialInstrument` share the single `'Yves Hilpisch'` string — no per-instance copy. By contrast, every instance has its own `symbol` and `price` in its own `__dict__`. So:

$$
\text{memory} \approx N_{\text{instances}} \times (\text{sizeof}(\text{instance dict}) + \text{sizeof}(\text{instance attrs}))
$$

For a million-instance simulation this can be the difference between a 200 MB and a 2 GB process. Use class attributes for *truly shared* metadata (constants, global state, methods themselves); use instance attributes for per-object data.

**Common bug to know:** mutable class attributes are shared by reference. If you do `class C: results = []` (class-level) and then `c1.results.append(1)`, `c2.results` will also contain `1`. Most of the time you want `self.results = []` inside `__init__`, which gives each instance its own list.


### 4.3 Encapsulation — getters, setters, and inheritance

A clean object exposes its internal state through methods, not direct attribute access. This pattern is called **encapsulation**. It lets you change the underlying storage later without breaking callers, and it gives you a single place to enforce invariants (e.g., "price must be positive").


In [21]:
# A new class that INHERITS from the previous FinancialInstrument
class FinancialInstrument(FinancialInstrument):
    """FinancialInstrument augmented with getter and setter methods."""

    def get_price(self):
        return self.price

    def set_price(self, price):
        # A setter is the natural place to enforce invariants
        # (we could check `if price < 0: raise ValueError(...)`).
        self.price = price


fi = FinancialInstrument('AAPL', 100)
print(f"Initial price (via getter): {fi.get_price()}")

# Use the setter to change price
fi.set_price(105)
print(f"After set_price(105):       {fi.get_price()}")

# Direct attribute access still works (and bypasses any validation we might add)
print(f"Direct fi.price access:     {fi.price}")

Initial price (via getter): 100
After set_price(105):       105
Direct fi.price access:     105


Two important details about this version:

**1. Inheritance via `class FinancialInstrument(FinancialInstrument)`.** This is a slightly mind-bending pattern: we *redefined* the name `FinancialInstrument` to refer to a new class that inherits from the old one. The old class is captured by the parent reference at definition time, so the inheritance chain works. In production code, you'd typically name them differently (`FinancialInstrumentV2`) or define everything in one block — this notebook style is a teaching device for showing incremental enhancement.

**2. Getters don't enforce anything yet.** As written, `fi.set_price(-50)` would happily set a negative price. The *value* of getters/setters comes from what you put inside them — validation, logging, lazy computation, caching. Python's culture is to delay introducing them until you need them; in Java they're written reflexively, but in Python the **`@property` decorator** lets you start with `self.price` and "upgrade" it later without changing the calling code. We'll show that briefly.

**Direct access still bypasses the getter.** This is what the next section addresses with name mangling.


### 4.4 "Private" attributes via name mangling

Python doesn't have truly private attributes. What it has is a **convention**: any attribute name starting with `__` (double underscore, no trailing underscore) gets *name-mangled* by the interpreter — `self.__price` is rewritten to `self._FinancialInstrument__price`. This makes accidental access from outside the class harder, but not impossible.


In [22]:
class FinancialInstrument(object):
    """FinancialInstrument with a 'private' (name-mangled) price attribute."""

    def __init__(self, symbol, price):
        self.symbol = symbol
        self.__price = price        # name will be mangled to _FinancialInstrument__price

    def get_price(self):
        return self.__price

    def set_price(self, price):
        self.__price = price


fi = FinancialInstrument('AAPL', 100)
print(f"Via getter:  {fi.get_price()}")

Via getter:  100


The instance now carries a private-by-convention price. The double underscore prefix triggers Python's **name mangling**: `__price` defined inside class `FinancialInstrument` is stored as `_FinancialInstrument__price`. The mangling is purely lexical — it happens at parse time based on the class name in the surrounding `class` block.

This explains why direct access fails:


In [23]:
# Direct access by the original name fails — the attribute doesn't exist by that name
fi.__price

AttributeError: 'FinancialInstrument' object has no attribute '__price'

**`AttributeError: 'FinancialInstrument' object has no attribute '__price'`** — exactly as expected. The interpreter looked up `__price` on the instance and didn't find it, because the actual stored attribute is named `_FinancialInstrument__price`. Inside the class methods, `self.__price` is *also* mangled to the same long name, so it resolves correctly. Outside the class, the unmangled name doesn't exist.

This is the value of name mangling: it makes outside access **non-trivial without making it impossible**.


In [24]:
# The name-mangled attribute IS accessible if you know the convention
print(f"Via mangled name: {fi._FinancialInstrument__price}")

# And we can write to it directly, bypassing the setter completely
fi._FinancialInstrument__price = 200
print(f"After direct mangled write: {fi.get_price()}")

# Restore via the proper API
fi.set_price(100)
print(f"After set_price(100):       {fi.get_price()}")

Via mangled name: 100
After direct mangled write: 200
After set_price(100):       100


**The Python culture around encapsulation:**

- Single underscore `_attr` — convention saying *"please don't touch this from outside, it's an implementation detail."* No mangling. Python doesn't enforce anything; it relies on the discipline of the caller. Linters and most IDEs warn you. Library authors use this for "internal-but-not-private" attributes.
- Double underscore `__attr` — name mangling kicks in. Strongest privacy you can get in Python, but as we've shown, even this isn't airtight.
- Double-underscore-double-trailing `__attr__` — **dunder** names. Reserved for Python's own data model (`__init__`, `__add__`, etc.). Don't invent your own.

**The Pythonic doctrine** is *"we are all consenting adults"* — you can break encapsulation if you really need to (debugging, monkey-patching, dependency-injection in tests), but the convention signals what's safe to rely on. Compare to languages like Java where `private` is enforced by the compiler: Python trades enforcement for flexibility. Most teams find this trade worthwhile, but it does mean you can't *trust* that an instance's state was reached only through approved methods.

**Modern alternative: `@property`.** In production code you usually shouldn't write `get_price`/`set_price` style getters and setters at all. Use `@property`:

```python
class FinancialInstrument:
    def __init__(self, symbol, price):
        self.symbol = symbol
        self._price = price

    @property
    def price(self):
        return self._price

    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("price must be non-negative")
        self._price = value

# Caller still writes fi.price = 105 (no method-call syntax),
# but the setter is invoked transparently.
```

This gives you encapsulation **without breaking the natural attribute-access syntax**. Adopt this in production; the explicit `get_price`/`set_price` pattern in the textbook is shown for pedagogy.


### 4.5 Aggregation — `PortfolioPosition`

A portfolio position holds *quantity* of a financial instrument. The `PortfolioPosition` class will *contain* a `FinancialInstrument` instance — this is **aggregation**: one object built from another that exists independently. The two-class design is more flexible than a single bloated class, because each component evolves on its own schedule.


In [25]:
class PortfolioPosition(object):
    """A holding of `position_size` units of a FinancialInstrument."""

    def __init__(self, financial_instrument, position_size):
        # Aggregation: this attribute is itself an object (a FinancialInstrument)
        self.position = financial_instrument
        # Private size attribute — same encapsulation pattern as before
        self.__position_size = position_size

    def get_position_size(self):
        return self.__position_size

    def update_position_size(self, position_size):
        self.__position_size = position_size

    def get_position_value(self):
        """Compute current market value: size × current price of the instrument."""
        return self.__position_size * self.position.get_price()


# Build a position of 10 shares of AAPL at $100/share
pp = PortfolioPosition(fi, 10)

print(f"position size:   {pp.get_position_size()}")
print(f"position value:  {pp.get_position_value()}    # 10 × $100 = $1000")
print(f"underlying instrument price: {pp.position.get_price()}")

position size:   10
position value:  1000    # 10 × $100 = $1000
underlying instrument price: 100


**Aggregation in action.** The `PortfolioPosition` does not duplicate the financial instrument's logic — it *references* an existing `FinancialInstrument` object via `self.position`. Updates to `fi`'s price are immediately reflected in `pp`'s computed value:


In [26]:
# Update the price of the underlying instrument directly
pp.position.set_price(105)

print(f"new instrument price: {pp.position.get_price()}")
print(f"new position value:   {pp.get_position_value()}    # 10 × $105 = $1050")

new instrument price: 105
new position value:   1050    # 10 × $105 = $1050


When the price changes from $100$ to $105$, the position value updates automatically: $10 \times 105 = 1050$. We did not have to tell `pp` that the price changed — it always asks `self.position` for the current price when computing the value.

**Why this matters in finance.** A portfolio of $N$ positions might share underlying instruments. If $\texttt{aapl}$ is held by 50 different positions and the price ticks up, all 50 see the update — *because they all reference the same `FinancialInstrument` instance, not copies of it*. This is referential semantics, and it's how realtime risk systems are typically built.

The contrast: if `PortfolioPosition` had stored `self.price = financial_instrument.price` (copying the value at construction time), each position would have its own stale price snapshot. You'd need an explicit "refresh" step to push updates through. **Aggregation by reference is what makes the design clean** — but it has its own pitfalls (mutability of shared state), which is why immutable value objects (think `dataclass(frozen=True)`) are increasingly popular for the data side.

**Aggregation vs. composition.** The textbook distinction:

- **Aggregation**: the contained object can exist independently. `aapl` exists as a `FinancialInstrument` whether or not any position holds it. Even after `pp` is garbage-collected, `aapl` lives on if other code references it.
- **Composition**: the contained object's lifetime is bound to the container. The two legs of a swap don't exist outside the swap. The container "owns" them.

In Python, the line is usually drawn by *who creates the inner object*: if `__init__` creates it (`self.legs = [Leg(...), Leg(...)]`), it's composition; if `__init__` accepts it as a parameter, it's aggregation.


## 5. The Python Data Model — Building `Vector`

The **Python data model** is the protocol of dunder methods (`__init__`, `__repr__`, `__add__`, `__len__`, `__iter__`, ...) that lets your custom classes interact with Python's built-in functions and operators. Implementing the right dunders makes your class *feel native*: `len(x)`, `for i in x`, `x + y`, `if x:`, `print(x)` — these all become available, with you in control of what they mean.

We'll build a 3-D `Vector` class step by step, adding one dunder at a time, and watch the class become more and more Pythonic. The mathematical motivation:

$$
\mathbf{v} = (x, y, z) \in \mathbb{R}^3
$$

with:

- Euclidean norm: $\|\mathbf{v}\| = \sqrt{x^2 + y^2 + z^2}$
- Vector addition: $(x_1, y_1, z_1) + (x_2, y_2, z_2) = (x_1+x_2, y_1+y_2, z_1+z_2)$
- Scalar multiplication: $\alpha \cdot (x, y, z) = (\alpha x, \alpha y, \alpha z)$

### 5.1 `__init__` — making the class instantiable


In [27]:
class Vector(object):
    def __init__(self, x=0, y=0, z=0):
        self.x = x
        self.y = y
        self.z = z


v = Vector(1, 2, 3)
v

The default `__repr__` gives us the unhelpful `<__main__.Vector at 0x...>`. The address tells us this is a *distinct* object (different from any other `Vector` instance), but tells us nothing about its values. For a class meant to be displayed often, this is a poor experience. We fix it next.


### 5.2 `__repr__` — controlling string representation


In [28]:
class Vector(Vector):
    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r}, {self.z!r})'


v = Vector(1, 2, 3)
print(f"repr(v):   {v!r}")
print(f"str(v):    {v}")
v

repr(v):   Vector(1, 2, 3)
str(v):    Vector(1, 2, 3)


Vector(1, 2, 3)

Now `Vector(1, 2, 3)` displays as `Vector(1, 2, 3)` — a representation that is **(a)** human-readable, **(b)** specific (we see the actual coordinates), and **(c)** ideally an expression you could `eval()` to reproduce the object.

**`__repr__` vs. `__str__`** — both control display, with subtly different roles:

- **`__repr__`**: for *developers*. Goal: unambiguous, ideally the source code that would recreate the object. Used by the interactive shell, debugger, and `repr(x)`.
- **`__str__`**: for *end users*. Goal: nicely formatted. Used by `print(x)` and `str(x)`.

If you only define one, define `__repr__` — Python falls back to `__repr__` when `__str__` is missing, but not vice versa. For a numeric class like `Vector`, the same string serves both audiences, so we leave `__str__` to fall through to `__repr__`.

The `!r` format spec (`{self.x!r}`) calls `repr()` on the value before formatting — important for strings (`'hello'` displays with quotes via `!r`, without via `!s`). For our integer/float coordinates the difference is invisible, but it's good practice to write `__repr__` defensively.


### 5.3 `__abs__` and `__bool__` — integrating with built-in functions

`abs(x)` calls `x.__abs__()`. `bool(x)` calls `x.__bool__()`. By implementing these, our `Vector` will work with the Python built-ins seamlessly.


In [29]:
class Vector(Vector):
    def __abs__(self):
        # The Euclidean norm of (x, y, z): sqrt(x^2 + y^2 + z^2)
        return (self.x ** 2 + self.y ** 2 + self.z ** 2) ** 0.5

    def __bool__(self):
        # A Vector is "true" iff it has nonzero magnitude
        return bool(abs(self))


# Nonzero vector
v = Vector(1, 2, -1)
print(f"abs(v):   {abs(v):.6f}    # sqrt(1 + 4 + 1) = sqrt(6)")
print(f"bool(v):  {bool(v)}")

# Zero vector
v_zero = Vector()
print(f"\nv_zero:           {v_zero}")
print(f"abs(v_zero):      {abs(v_zero)}")
print(f"bool(v_zero):     {bool(v_zero)}")

abs(v):   2.449490    # sqrt(1 + 4 + 1) = sqrt(6)
bool(v):  True

v_zero:           Vector(0, 0, 0)
abs(v_zero):      0.0
bool(v_zero):     False


**`abs(v)` returns the Euclidean norm.** For `Vector(1, 2, -1)`:

$$
\|\mathbf{v}\| = \sqrt{1^2 + 2^2 + (-1)^2} = \sqrt{6} \approx 2.4495
$$

The output `2.449489742783178` matches to full float precision. ✓

**`bool(v)` returns `True` for nonzero vectors, `False` for the zero vector.** Why does this matter? Because Python uses `__bool__` *implicitly* in conditional contexts:

```python
v = Vector(0, 0, 0)
if v:                    # calls bool(v), which is False
    print('non-zero')
else:
    print('zero')        # this branch fires
```

Without `__bool__`, Python falls back to `__len__()` (we'll implement that next) and treats the object as truthy if `len > 0`. Without either, every instance is truthy by default. **Defining `__bool__` is what makes `if v:` mean what you expect** — it's the difference between "is this vector zero?" (your semantics) and "does this vector exist?" (Python's default).

For `Vector()` (the zero vector): the norm is exactly $0.0$, and `bool(0.0)` is `False`. ✓

**Numerical caveat for production:** comparing floats to zero exactly is rarely what you want for floating-point vectors that came from real computations. The vector `Vector(1e-300, 0, 0)` has norm $10^{-300}$ — technically nonzero, mathematically zero for any practical purpose. A robust `__bool__` for production would compare against an epsilon: `return abs(self) > 1e-12`. The textbook version is correct for integer or symbolic coordinates; in floating-point pipelines, threshold it.


### 5.4 `__add__` and `__mul__` — operator overloading

The `+` and `*` operators dispatch to `__add__` and `__mul__`. Implementing these lets us write vector arithmetic naturally.


In [30]:
class Vector(Vector):
    def __add__(self, other):
        # Vector addition — return a new Vector (don't mutate self!)
        return Vector(self.x + other.x,
                      self.y + other.y,
                      self.z + other.z)

    def __mul__(self, scalar):
        # Scalar multiplication
        return Vector(self.x * scalar,
                      self.y * scalar,
                      self.z * scalar)


v = Vector(1, 2, 3)
print(f"v:                       {v}")
print(f"v + Vector(2, 3, 4):     {v + Vector(2, 3, 4)}")
print(f"v * 2:                   {v * 2}")

v:                       Vector(1, 2, 3)
v + Vector(2, 3, 4):     Vector(3, 5, 7)
v * 2:                   Vector(2, 4, 6)


Two key design decisions here:

**1. Operations return new objects, they don't mutate `self`.** `v + w` does not change `v`; it returns a fresh `Vector`. This is **value semantics** — it's how `int`, `float`, `str`, and `tuple` work in Python. Most numerical types should follow the same convention. Mutation creates aliasing bugs (`a = b; a += c` accidentally changes `b` too).

**2. The implementation is concrete, not generic.** Our `__add__` requires `other` to have `.x`, `.y`, `.z` attributes — it would crash on `v + 5` or `v + np.array([1, 2, 3])`. A production-grade implementation would accept a few input types and dispatch:

```python
def __add__(self, other):
    if isinstance(other, Vector):
        return Vector(self.x + other.x, ...)
    elif isinstance(other, (int, float)):
        return Vector(self.x + other, ...)  # broadcast
    else:
        return NotImplemented   # let Python try `other.__radd__(self)`
```

Returning `NotImplemented` (the singleton, capital N) signals to Python "I don't know how to add this; please try the other operand's `__radd__`." This is how libraries cooperate without knowing about each other.

For our pedagogical version, simple is better. The math is correct: $(1, 2, 3) + (2, 3, 4) = (3, 5, 7)$ ✓ and $(1, 2, 3) \cdot 2 = (2, 4, 6)$ ✓.


### 5.5 `__len__` and `__getitem__` — collection-like behavior

A 3-D vector has 3 coordinates. By implementing `__len__` and `__getitem__`, we make our `Vector` behave like a sequence — `len(v)` works, `v[0]` works, and as a bonus we get *iteration* and `in` checks for free.


In [31]:
class Vector(Vector):
    def __len__(self):
        return 3

    def __getitem__(self, i):
        # Support both forward (0, 1, 2) and reverse (-1, -2, -3) indexing
        if i in (0, -3):
            return self.x
        elif i in (1, -2):
            return self.y
        elif i in (2, -1):
            return self.z
        else:
            raise IndexError('Index out of range.')


v = Vector(1, 2, 3)
print(f"len(v):      {len(v)}")
print(f"v[0]:        {v[0]}")
print(f"v[1]:        {v[1]}")
print(f"v[2]:        {v[2]}")
print(f"v[-1]:       {v[-1]}    # negative indexing: last element")
print(f"v[-2]:       {v[-2]}")

len(v):      3
v[0]:        1
v[1]:        2
v[2]:        3
v[-1]:       3    # negative indexing: last element
v[-2]:       2


Now `Vector` walks like a sequence. The `__getitem__` method handles **forward indexing** (`0, 1, 2` → `x, y, z`) and **negative indexing** (`-1, -2, -3` → `z, y, x`, counting from the end) — exactly the Python convention. The conditional structure ensures both indexes for the same coordinate land in the same branch (e.g., `0` and `-3` both return `self.x`).

For `Vector(1, 2, 3)`:

- `v[0] = v[-3] = 1` (the `x`)
- `v[1] = v[-2] = 2` (the `y`)
- `v[2] = v[-1] = 3` (the `z`)

All consistent.


In [32]:
# Out-of-range index raises IndexError — Python's expected behavior for sequences
v[3]

IndexError: Index out of range.

**`IndexError: Index out of range.`** — exactly as expected and as Python convention demands. Any sequence-like type that doesn't raise `IndexError` for out-of-bounds access will misbehave inside `for` loops, list-comprehensions, and the rest of Python's iteration machinery (which uses `IndexError` as the implicit "stop" signal when iterating via `__getitem__`).

This cooperation between exception types and iteration is one of the many subtle protocols of the data model. By raising the *right* exception, we make our class plug into the rest of Python.


### 5.6 `__iter__` — making the class a true iterable

Even without `__iter__`, our class is *iterable* via the `__getitem__` fallback (Python tries `obj[0], obj[1], ...` until `IndexError`). But explicit `__iter__` is faster and more idiomatic.


In [33]:
class Vector(Vector):
    def __iter__(self):
        # A generator — yields each coordinate in turn
        for i in range(len(self)):
            yield self[i]


v = Vector(1, 2, 3)

# Indirect iteration (via __getitem__, was already working)
print("Loop via index:")
for i in range(len(v)):
    print(f"  v[{i}] = {v[i]}")

# Direct iteration (via __iter__, the proper way)
print("\nLoop via iter:")
for coord in v:
    print(f"  {coord}")

# Iterables work with all of Python's iteration tooling
print(f"\nlist(v):     {list(v)}")
print(f"tuple(v):    {tuple(v)}")
print(f"sum(v):      {sum(v)}              # 1 + 2 + 3")
print(f"max(v):      {max(v)}")
print(f"3 in v:      {3 in v}")

Loop via index:
  v[0] = 1
  v[1] = 2
  v[2] = 3

Loop via iter:
  1
  2
  3

list(v):     [1, 2, 3]
tuple(v):    (1, 2, 3)
sum(v):      6              # 1 + 2 + 3
max(v):      3
3 in v:      True


**`__iter__` returns a generator** (because we `yield` instead of `return`). Each `yield` produces the next value and pauses the function; the next iteration resumes where we left off. This is a *lazy* iteration — the values are produced one at a time, not pre-computed into a list. For a 3-element vector that's irrelevant, but for a class wrapping millions of records, it's the difference between fitting in RAM and not.

Once `__iter__` is in place, our `Vector` plugs into **the entire ecosystem of iterable consumers**:

- `for x in v:` (the `for` loop)
- `list(v)`, `tuple(v)` (constructors that take iterables)
- `sum(v)`, `min(v)`, `max(v)` (reductions over iterables — `sum(v) = 1+2+3 = 6`)
- `3 in v` (the `in` operator falls back to iteration when `__contains__` isn't defined)
- `enumerate(v)`, `zip(v, w)`, `map(f, v)`, `filter(p, v)` (composable iterators)
- list/dict/set comprehensions: `[x**2 for x in v]`
- the `*` unpacking operator: `f(*v)` calls `f(1, 2, 3)`

**This is the payoff of the data model.** Twenty lines of class code earned us interaction with hundreds of standard library functions and language constructs. Compare to languages where every collection requires its own iteration interface defined explicitly — Python's *protocols-over-interfaces* approach is one of the major reasons for the language's expressiveness.

**Connection to ML:** PyTorch's `Dataset` class implements `__len__` and `__getitem__`. That's it. From those two methods, the entire data-loading machinery (`DataLoader`, batching, shuffling, multi-process workers) becomes available. The same protocol underlies pandas's row iteration, scikit-learn's CV splitters, and HuggingFace's `datasets`. **Learn this protocol once and it pays off everywhere.**


## 6. The Complete `Vector` Class

Putting it all together — one class, all the dunders we've added, fully self-contained.


In [34]:
class Vector(object):
    """A 3-dimensional vector with full Python data-model support.

    Demonstrates: __init__, __repr__, __abs__, __bool__,
                  __add__, __mul__, __len__, __getitem__, __iter__
    """

    def __init__(self, x=0, y=0, z=0):
        self.x = x
        self.y = y
        self.z = z

    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r}, {self.z!r})'

    def __abs__(self):
        return (self.x ** 2 + self.y ** 2 + self.z ** 2) ** 0.5

    def __bool__(self):
        return bool(abs(self))

    def __add__(self, other):
        return Vector(self.x + other.x,
                      self.y + other.y,
                      self.z + other.z)

    def __mul__(self, scalar):
        return Vector(self.x * scalar,
                      self.y * scalar,
                      self.z * scalar)

    def __len__(self):
        return 3

    def __getitem__(self, i):
        if i in (0, -3): return self.x
        elif i in (1, -2): return self.y
        elif i in (2, -1): return self.z
        else: raise IndexError('Index out of range.')

    def __iter__(self):
        for i in range(len(self)):
            yield self[i]


# A small showcase
u = Vector(1, 2, 3)
v = Vector(4, 5, 6)
print(f"u                 = {u}")
print(f"v                 = {v}")
print(f"u + v             = {u + v}")
print(f"u * 3             = {u * 3}")
print(f"|u|               = {abs(u):.6f}")
print(f"bool(u)           = {bool(u)}")
print(f"len(u)            = {len(u)}")
print(f"u[0], u[1], u[2]  = {u[0]}, {u[1]}, {u[2]}")
print(f"sum(u)            = {sum(u)}")
print(f"list(u)           = {list(u)}")
print(f"max(u)            = {max(u)}")
print(f"2 in u            = {2 in u}")
print(f"99 in u           = {99 in u}")

u                 = Vector(1, 2, 3)
v                 = Vector(4, 5, 6)
u + v             = Vector(5, 7, 9)
u * 3             = Vector(3, 6, 9)
|u|               = 3.741657
bool(u)           = True
len(u)            = 3
u[0], u[1], u[2]  = 1, 2, 3
sum(u)            = 6
list(u)           = [1, 2, 3]
max(u)            = 3
2 in u            = True
99 in u           = False


Every operation produced the expected result:

- $\mathbf{u} + \mathbf{v} = (1+4, 2+5, 3+6) = (5, 7, 9)$ ✓
- $3 \mathbf{u} = (3, 6, 9)$ ✓
- $\|\mathbf{u}\| = \sqrt{1 + 4 + 9} = \sqrt{14} \approx 3.7417$ ✓
- $\mathrm{sum}(\mathbf{u}) = 1 + 2 + 3 = 6$ ✓
- `2 in u` is `True` because `2 == u[1]`; `99 in u` is `False` ✓

In about 30 lines, we built a class that behaves consistently with every Python language construct an experienced developer would expect to work. **No external library involvement, no inheritance from a special base class, no interface declarations** — just the data-model dunders.

### What we left out (and what to add for production)

A real-world `Vector` class would also implement:

- `__eq__` and `__hash__` — for use in sets and dict keys.
- `__sub__`, `__neg__` — vector subtraction and negation.
- `__rmul__` — to make `2 * v` work (right now `v * 2` works because `Vector.__mul__` is called; `2 * v` would try `int.__mul__(2, v)` first, which doesn't know about `Vector`, then fall back to `Vector.__rmul__`).
- `__truediv__` — division by a scalar.
- `__matmul__` — the `@` operator, naturally for dot products: `u @ v = u.x*v.x + u.y*v.y + u.z*v.z`.
- `__format__` — control over `f'{v:.2f}'`-style formatting.

For numerical work, **the bigger lesson is to use `numpy.ndarray` (or `dataclass(frozen=True)` wrapping it) rather than rolling your own.** A NumPy array gives you all of the above plus broadcasting, ufuncs, BLAS routines, and 1000$\times$ better performance for any nontrivial size. Custom classes shine when the *semantics* are unique enough that NumPy doesn't suffice — `Position`, `Trade`, `OrderBook`, `Strategy`, `Tensor`, `Distribution`. Each of these has invariants and methods that go well beyond "an array of numbers."


## 7. Conclusion

Object-oriented programming is the Python paradigm — every value, every function, every module is an object. We saw three escalating levels of OOP fluency:

**Level 1: using objects.** Calling `int.bit_length()`, `list.append()`, `df.cumsum()`, `np.sum(arr)` — recognizing that the dot syntax and operator behavior of every Python value is *defined by* a class. This level is enough to be a productive consumer of libraries.

**Level 2: writing classes.** Defining `__init__`, instance and class attributes, methods, and using inheritance and composition. The `FinancialInstrument` $\to$ `PortfolioPosition` example showed how a few well-chosen classes model a domain naturally — and how aggregation by reference enables real-time consistency between objects.

**Level 3: integrating with Python's data model.** Implementing dunders so your classes interact with operators (`__add__`, `__mul__`), built-in functions (`__len__`, `__abs__`, `__bool__`), iteration (`__iter__`, `__getitem__`), and string display (`__repr__`). The `Vector` class showed how 30 lines of dunder methods plug a custom type into the entire Python language and standard library.

### Connections forward

This OOP machinery underlies essentially every framework you'll meet in the rest of an ML/quant career:

- **scikit-learn**: `BaseEstimator` defines the `fit/predict/score` contract; mixins (`ClassifierMixin`, `RegressorMixin`) add typed behavior; pipelines compose estimators via aggregation.
- **PyTorch**: `nn.Module` overloads `__call__` to fuse input-validation with `forward()`; sub-modules are tracked through `__setattr__` introspection; tensors define `__add__` etc. with autograd as a side effect.
- **pandas**: `DataFrame` and `Series` implement nearly the entire data model — operators, iteration, hashing, descriptive `__repr__` — which is why analytical code reads so naturally.
- **dataclasses & pydantic**: code-generation patterns that write `__init__`, `__repr__`, `__eq__`, validation logic for you, so you can focus on the domain semantics.

### When to reach for OOP — and when not

OOP is a **tool**, not a religion. Use a class when:

- you have related data + behaviors that should live together;
- the abstraction will be instantiated more than once with different state;
- there's a clear interface other code will want to depend on.

Use a plain function or a `dataclass` (or even a tuple) when:

- you just need to bundle a few values with no behavior;
- the "class" you're tempted to write only ever has one method (just write the function);
- inheritance hierarchies start feeling forced — composition is almost always cleaner.

Pamela Zave's epigraph captures the right mindset: software engineering's job is to **control complexity, not create it**. OOP is one of the best tools for the former; used carelessly, it's a great way to do the latter.
